# OpenPlaque — Proximal LAD Long-Reach to Aortic Root v1.2
Feasibility correction to the prior calibrated LAD→aorta experiment. The prior accepted LAD endpoint was ~36 mm from the aortic surface but the search was capped at 20 mm. This run keeps the prior source/mask gates unchanged and prospectively extends only the left reachability budget.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/Left_Main_Proximal_LAD_Long_Reach_v1_2'
BRANCH = 'left-main-proximal-lad-long-reach-from-main'
PINNED_SCIENCE_COMMIT = '2d9305112f2e2d8ae9d44cf0df2e725d09d959f0'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM = 'left-main-proximal-lad-long-reach-v1.2-feasible'
EXPECTED_PREREQ_ALGORITHM = 'left-coronary-source-ostium-multiseed-v2.1-control-adjudication'
print('Branch:', BRANCH)
print('Pinned science commit:', PINNED_SCIENCE_COMMIT)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 20 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout --detach $PINNED_SCIENCE_COMMIT
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)
assert HEAD == PINNED_SCIENCE_COMMIT, (HEAD, PINNED_SCIENCE_COMMIT)
merge_base = !git -C /content/OpenPlaque merge-base HEAD $BASELINE
print('Merge base:', merge_base[0] if merge_base else 'missing')
assert merge_base and merge_base[0].strip() == BASELINE


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_main_proximal_lad_long_reach_v1_2 as exp
print('openplaque:', openplaque.__file__)
print('experiment:', exp.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(), exp.__file__, 'exec')
print('synthetic reachability:', exp.synthetic_reachability_self_test())
test_file='/content/OpenPlaque/tests/test_left_main_proximal_lad_long_reach_v1_2.py'
rc = pytest.main(['-q', test_file])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
import json
root=Path(DRIVE_ROOT)
required=[
 root/'Left_Coronary_Source_Ostium_Multiseed_Control_Adjudication_v2_1/summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
prior=json.loads(required[0].read_text())
print('Prerequisite status:', prior.get('status'))
print('Prerequisite algorithm:', prior.get('algorithm'))
print('Prerequisite RCA control pass:', prior.get('RCA_control_pass'))
assert prior.get('baseline_commit') == BASELINE
assert prior.get('algorithm') == EXPECTED_PREREQ_ALGORITHM
assert prior.get('RCA_control_pass') is True


In [ ]:
import gc, time
from openplaque.left_main_proximal_lad_long_reach_v1_2 import run
gc.collect()
t0=time.time()
result = run(DRIVE_ROOT, OUTPUT_DIR)
s=result['summary']
budget=s.get('left_search_reachability_budget',{})
left=s.get('proximal_LAD_long_reach',{})
print('ELAPSED MIN:', round((time.time()-t0)/60,2))
print('STATUS:', s['status'])
print('ACCEPTED LAD ENDPOINT→AORTA DISTANCE MM:', budget.get('initial_aorta_distance_mm'))
print('LONG-REACH SEARCH BUDGET MM:', budget.get('max_search_mm'))
print('BUDGET SLACK MM:', budget.get('budget_minus_straight_line_mm'))
print('RCA CONTROL PASS:', s.get('RCA_calibrated_proximal_retrace',{}).get('accepted'))
print('LEFT SOURCE GATE PASS:', s.get('left_source_gate_pass'))
print('RCA-INDEPENDENT AORTIC ENDPOINT:', s.get('posthoc_RCA_independent'))
print('LEFT RESULT:', left)
print('REPORT:', result.get('report'))
print('ZIP:', result.get('zip'))
